# Supplemental figure: p-value and circular $R^2$ of each trajectory's own fit

Makes `own_fit_summary.svg` (this folder) and the LaTeX table `output_figs/own_fit_r2_table.tex`. Moved here from `Figure_5_option/`; column headings, column order and colors match Figure 5 panel E (`FigureGeneration/Main/fig_5_unifying_algo_analysis/generate_figure_5/r2_distribution_summary.ipynb`).

Same datasets and column layout as Figure 5 panel E, but each trajectory is scored against **its own best unifying-algorithm fit** (lowest `rmse_affine`), not the mean/parametric model. Both values are computed over that fit's own ROI (`min_ix:max_ix`), i.e. the samples the fit was optimized on.

- **Row A: $p$-value** of the Jammalamadaka–SenGupta circular correlation between the course and its own fit. The correlation coefficient is the `rho` column stored in the unifying results parquets (recomputed here, matches to ~1e-14); the parquets store no p-value, so it is computed from rho in `own_fit_helper.py`. Plotted as $\log_{10} p$; gray line at $p = 0.05$.
- **Row B: circular $R^2$** (`uaa.circular_r2`) of the course vs. its own fit (not stored in the parquets; rebuilt from the stored fit parameters). Linear axis; gray line at 0.

**Interpretation caveat.** The test assumes ~200 independent samples, but they are 2 s of a smoothed course at 100 Hz, and the fit (slope, intercept, 4–6 affine parameters, chosen as the best of the bootstrap iterations) was optimized on exactly these samples. So both numbers are in-sample and the p-values are much too small in an absolute sense. Compare flash with the sham, shuffled and synthetic controls rather than with $p = 0.05$ or $R^2 = 0$.

Values outside the axis limits are clipped to the limit (counts printed as 'outside limits'). Each column is one figurefirst figure with two axes (`ax_p`, `ax_r2`) in `own_fit_summary.svg`. Run **Setup** first; each section then runs on its own, except **Table**, which needs all four compute cells.

## Setup

In [1]:
import matplotlib
matplotlib.rcParams['font.serif'] = ['Times'] + matplotlib.rcParams['font.serif']  # same as Figure 5 notebooks
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

from splitflow.set_zorder_functions import set_selective_rasterization

import slope_r2_helper as h
import own_fit_helper as of

FIGURE_NAME = of.FIGURE_NAME
TRANSLATION = h.TRANSLATION


def add_group_labels(ax, groups, y=-0.45):
    """Text under the tick labels, centered on each group of columns. groups: list of (label, [x positions])."""
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    for label, xs in groups:
        ax.text(np.mean(xs), y, label, transform=trans, ha='center', va='top', fontsize=6)


def load_axes(fifi_figure):
    layout = h.load_layout(FIGURE_NAME)
    return layout, layout.axes[(fifi_figure, 'ax_p')], layout.axes[(fifi_figure, 'ax_r2')]


def finish(layout, fifi_figure, axes):
    for ax in axes:
        set_selective_rasterization(ax, rasterize_markers=['o'])
    h.save_figure(layout, fifi_figure, FIGURE_NAME)

/Users/florisvb/src/SplitFlowOptoTracking/.venv/lib/python3.11/site-packages/figurefirst/mpl_fig_to_figurefirst_svg.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cpu


## Flash: still air / unsteady / laminar (40 cm/s)


In [2]:
FIFI_FIGURE = 'own_fit_fig5_flash'

PANEL1 = {'stillair': ('stillair_data_flash_translation%s.parquet' % TRANSLATION,        '#084a72', 'Still air'),
          'unsteady': ('unsteady_top_on_data_flash_translation%s.parquet' % TRANSLATION, '#2b75b3', 'Unsteady'),
          'laminar':  ('laminar_data_flash_translation%s.parquet' % TRANSLATION,        '#991128', 'Laminar\n(40 cm/s)')}

panel1_results = {}
for windtype, (fname, color, _) in PANEL1.items():
    unifying_filename = h.unifying_results_path(fname)
    panel1_results[windtype] = of.compute_own_fit_stats(unifying_filename, h.fly_trajec_filename(unifying_filename),
                                                        label='flash ' + windtype)

flash stillair               n =  200 (dropped 0)  median R2 = 0.91  median p = 10^-25.2  frac p<0.05 = 0.99  outside limits: 0  max|rho - stored| = 7.6e-15


flash unsteady               n =   60 (dropped 0)  median R2 = 0.87  median p = 10^-23.3  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 1.1e-15


flash laminar                n =  145 (dropped 0)  median R2 = 0.87  median p = 10^-29.6  frac p<0.05 = 0.98  outside limits: 0  max|rho - stored| = 6.3e-14


In [3]:
layout, ax_p, ax_r2 = load_axes(FIFI_FIGURE)
np.random.seed(h.RNG_SEED)  # jitter and bootstrap CI

xs = np.arange(1, len(PANEL1) + 1)
for x, (windtype, (_, color, _)) in zip(xs, PANEL1.items()):
    of.plot_column(ax_p, ax_r2, x, panel1_results[windtype], color)

of.clean_column_labels(ax_p, ax_r2, xs, [v[2] for v in PANEL1.values()], xlim=(0.5, len(PANEL1) + 0.5),
                       rotation=45, ticklabel_colors=[v[1] for v in PANEL1.values()])
finish(layout, FIFI_FIGURE, [ax_p, ax_r2])

## Flash: variable wind (all speeds)

Colors as in Figure 5: light purple, 20 cm/s highlighted.


In [4]:
FIFI_FIGURE = 'own_fit_varwind_flash'
PANEL2_COLORS = {speed: (h.VARWIND_HIGHLIGHT_COLOR if speed == h.VARWIND_HIGHLIGHT_SPEED else h.VARWIND_COLOR)
                 for speed in h.VARWIND_SPEEDS}  # as in Figure 5

panel2_results = {}
for speed in h.VARWIND_SPEEDS:
    panel2_results[speed] = of.compute_own_fit_stats(h.varwind_unifying_filename('flash', speed), h.VARWIND_TRAJEC_FILENAME,
                                                     label='var wind flash %d cm/s' % h.VARWIND_SPEED_LABELS[speed])

var wind flash 0 cm/s        n =  700 (dropped 0)  median R2 = 0.95  median p = 10^-26.4  frac p<0.05 = 0.99  outside limits: 0  max|rho - stored| = 1.2e-15


var wind flash 3 cm/s        n =  491 (dropped 0)  median R2 = 0.93  median p = 10^-26.3  frac p<0.05 = 0.98  outside limits: 0  max|rho - stored| = 7.8e-16


var wind flash 10 cm/s       n =  257 (dropped 0)  median R2 = 0.90  median p = 10^-26.5  frac p<0.05 = 1.00  outside limits: 0  max|rho - stored| = 2.7e-15


var wind flash 20 cm/s       n =  700 (dropped 0)  median R2 = 0.90  median p = 10^-27.0  frac p<0.05 = 0.98  outside limits: 0  max|rho - stored| = 4.0e-14


var wind flash 30 cm/s       n =  272 (dropped 0)  median R2 = 0.88  median p = 10^-27.2  frac p<0.05 = 0.99  outside limits: 0  max|rho - stored| = 1.0e-15


var wind flash 60 cm/s       n =  241 (dropped 0)  median R2 = 0.90  median p = 10^-31.1  frac p<0.05 = 0.98  outside limits: 0  max|rho - stored| = 1.2e-15


In [5]:
layout, ax_p, ax_r2 = load_axes(FIFI_FIGURE)
np.random.seed(h.RNG_SEED)

xs = np.arange(1, len(h.VARWIND_SPEEDS) + 1)
for x, speed in zip(xs, h.VARWIND_SPEEDS):
    of.plot_column(ax_p, ax_r2, x, panel2_results[speed], PANEL2_COLORS[speed])

of.clean_column_labels(ax_p, ax_r2, xs, [str(h.VARWIND_SPEED_LABELS[s]) for s in h.VARWIND_SPEEDS],
                       xlim=(0.5, len(xs) + 0.5), show_ylabel=False, xlabel='Wind speed (cm/s)')
finish(layout, FIFI_FIGURE, [ax_p, ax_r2])

## Sham, WT flash, WT sham

Same four groups as Figure 5 panel E (headings in the SVG: Sham = variable-wind sham + sham still air/unsteady/laminar).


In [6]:
FIFI_FIGURE = 'own_fit_sham_wt'
GROUP_GAP = 0.75
FLY_TICKLABELS = {'stillair': 'Still air', 'unsteady': 'Unsteady', 'laminar': 'Laminar'}

PANEL3_FLIES = {
    'Sham': {'stillair': ('stillair_data_sham_translation%s.parquet' % TRANSLATION,                    'gray'),
             'unsteady': ('unsteady_top_on_data_sham_AUGMENTED_translation%s.parquet' % TRANSLATION,   'gray'),
             'laminar':  ('laminar_data_sham_translation%s.parquet' % TRANSLATION,                     'gray')},
    'WT flash': {'stillair': ('WT_stillair_data_flash_translation%s.parquet' % TRANSLATION,                    'gray'),
                 'unsteady': ('experiencedunsteady_top_on_data_WT_flash_translation%s.parquet' % TRANSLATION,  'gray'),
                 'laminar':  ('WT_laminar_data_flash_translation%s.parquet' % TRANSLATION,                     'gray')},
    'WT sham': {'stillair': ('WT_stillair_data_sham_translation%s.parquet' % TRANSLATION,                    'gray'),
                'unsteady': ('experiencedunsteady_top_on_data_WT_sham_translation%s.parquet' % TRANSLATION,  'gray'),
                'laminar':  ('WT_laminar_data_sham_translation%s.parquet' % TRANSLATION,                     'gray')},
}
PANEL3_VARWIND_SHAM_COLORS = {speed: 'gray' for speed in h.VARWIND_SPEEDS}

panel3_results = {'Var. wind sham (cm/s)': []}
for speed in h.VARWIND_SPEEDS:
    r = of.compute_own_fit_stats(h.varwind_unifying_filename('sham', speed), h.VARWIND_TRAJEC_FILENAME,
                                 label='var wind sham %d cm/s' % h.VARWIND_SPEED_LABELS[speed])
    panel3_results['Var. wind sham (cm/s)'].append((str(h.VARWIND_SPEED_LABELS[speed]), r, PANEL3_VARWIND_SHAM_COLORS[speed]))
for group, datasets in PANEL3_FLIES.items():
    panel3_results[group] = []
    for windtype, (fname, color) in datasets.items():
        unifying_filename = h.unifying_results_path(fname)
        r = of.compute_own_fit_stats(unifying_filename, h.fly_trajec_filename(unifying_filename), label=group + ' ' + windtype)
        panel3_results[group].append((FLY_TICKLABELS[windtype], r, color))

var wind sham 0 cm/s         n =  388 (dropped 0)  median R2 = 0.78  median p = 10^-20.4  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 1.2e-14


var wind sham 3 cm/s         n =  423 (dropped 0)  median R2 = 0.79  median p = 10^-21.1  frac p<0.05 = 0.92  outside limits: 0  max|rho - stored| = 4.4e-15


var wind sham 10 cm/s        n =  241 (dropped 0)  median R2 = 0.79  median p = 10^-20.5  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 7.8e-16


var wind sham 20 cm/s        n =  225 (dropped 0)  median R2 = 0.76  median p = 10^-19.7  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 7.1e-13


var wind sham 30 cm/s        n =  232 (dropped 0)  median R2 = 0.74  median p = 10^-19.2  frac p<0.05 = 0.91  outside limits: 0  max|rho - stored| = 2.6e-15


var wind sham 60 cm/s        n =   68 (dropped 0)  median R2 = 0.71  median p = 10^-13.7  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 5.6e-16


... (2 outputs left out)

WT flash unsteady            n =   68 (dropped 0)  median R2 = 0.75  median p = 10^-18.6  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 4.4e-16


WT flash laminar             n =  138 (dropped 0)  median R2 = 0.64  median p = 10^-17.0  frac p<0.05 = 0.97  outside limits: 0  max|rho - stored| = 1.9e-15
WT sham stillair             n =   68 (dropped 0)  median R2 = 0.59  median p = 10^-17.3  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 5.8e-15


WT sham unsteady             n =   66 (dropped 0)  median R2 = 0.73  median p = 10^-21.5  frac p<0.05 = 0.95  outside limits: 0  max|rho - stored| = 3.6e-16


WT sham laminar              n =  127 (dropped 0)  median R2 = 0.62  median p = 10^-17.8  frac p<0.05 = 0.98  outside limits: 0  max|rho - stored| = 2.8e-15


In [7]:
layout, ax_p, ax_r2 = load_axes(FIFI_FIGURE)
np.random.seed(h.RNG_SEED)

xticks, xticklabels, group_positions = [], [], []
x = 1.0
for group, columns in panel3_results.items():
    xs_group = []
    for ticklabel, r, color in columns:
        of.plot_column(ax_p, ax_r2, x, r, color)
        xticks.append(x); xticklabels.append(ticklabel); xs_group.append(x)
        x += 1
    group_positions.append((group, xs_group))
    x += GROUP_GAP

of.clean_column_labels(ax_p, ax_r2, xticks, xticklabels, xlim=(0.5, xticks[-1] + 0.5), show_ylabel=False, rotation=45)
add_group_labels(ax_r2, group_positions)
finish(layout, FIFI_FIGURE, [ax_p, ax_r2])

## Shuffled & Synthetic


In [8]:
FIFI_FIGURE = 'own_fit_shuffled_synthetic'

scenarios = h.shuffled_synthetic_scenarios()
PANEL4_COLORS = {key: 'gray' for key in scenarios.SCENARIOS}

panel4_results = {}
for key in scenarios.SCENARIOS:
    unifying_filename, trajec_filename = h.shuffled_synthetic_filenames(key)
    panel4_results[key] = of.compute_own_fit_stats(unifying_filename, trajec_filename, label=key)

shuffled_laminar             n =  200 (dropped 0)  median R2 = 0.86  median p = 10^-25.1  frac p<0.05 = 0.97  outside limits: 0  max|rho - stored| = 0.0e+00


shuffled_stillair            n =  200 (dropped 0)  median R2 = 0.86  median p = 10^-25.8  frac p<0.05 = 0.97  outside limits: 0  max|rho - stored| = 0.0e+00


levy_flight                  n =  200 (dropped 0)  median R2 = 0.03  median p = 10^-2.7  frac p<0.05 = 0.65  outside limits: 0  max|rho - stored| = 0.0e+00


random_turns                 n =  200 (dropped 0)  median R2 = 0.72  median p = 10^-21.3  frac p<0.05 = 0.96  outside limits: 0  max|rho - stored| = 0.0e+00


brownian                     n =  200 (dropped 0)  median R2 = 0.32  median p = 10^-5.8  frac p<0.05 = 0.95  outside limits: 0  max|rho - stored| = 0.0e+00


run_and_tumble               n =  200 (dropped 0)  median R2 = 0.34  median p = 10^-6.2  frac p<0.05 = 0.93  outside limits: 0  max|rho - stored| = 0.0e+00


In [9]:
layout, ax_p, ax_r2 = load_axes(FIFI_FIGURE)
np.random.seed(h.RNG_SEED)

xs = np.arange(1, len(scenarios.SCENARIOS) + 1)
for x, key in zip(xs, scenarios.SCENARIOS):
    of.plot_column(ax_p, ax_r2, x, panel4_results[key], PANEL4_COLORS[key])

of.clean_column_labels(ax_p, ax_r2, xs, [scenarios.SHORT_LABELS[k] for k in scenarios.SCENARIOS],
                       xlim=(0.5, len(xs) + 0.5), show_ylabel=False, rotation=45)
finish(layout, FIFI_FIGURE, [ax_p, ax_r2])

## Table: median own-fit $R^2$ with 95% CI

One row per dataset (all columns of the figure). The 95% CI of the median is the same bootstrap the plots shade
(`flystat.resampling.bootstrap_confidence_intervals_from_data`, `use='median'`), on unclipped values.

The caption gives mean ± SD over pooled trajectories for two groups:
- **Fictive odor**: all flash datasets with Chrimson flies (still air, unsteady, laminar, and the six variable-wind speeds).
- **Controls**: variable-wind sham, sham still air/unsteady/laminar, WT flash, and WT sham.

$p$ is summarized as mean ± SD of $\log_{10} p$.


In [10]:
import pandas as pd
import flystat.resampling

TABLE_FILENAME = 'output_figs/own_fit_r2_table.tex'

# (group, condition, results dict); order as in the figure
table_rows = [('Flash', v[2].replace('\n', ' '), panel1_results[k]) for k, v in PANEL1.items()]
table_rows += [('Flash, variable wind', '%d cm/s' % h.VARWIND_SPEED_LABELS[s], panel2_results[s]) for s in h.VARWIND_SPEEDS]
SHAM_GROUP_NAMES = {'Var. wind sham (cm/s)': 'Sham, variable wind', 'Sham': 'Sham', 'WT flash': 'WT flash', 'WT sham': 'WT sham'}
for group, columns in panel3_results.items():
    for ticklabel, r, _ in columns:
        cond = ticklabel + ' cm/s' if group.startswith('Var. wind') else ticklabel
        table_rows.append((SHAM_GROUP_NAMES[group], cond, r))
table_rows += [('Shuffled \\& synthetic', scenarios.SHORT_LABELS[k].replace('\n', ' ').replace('&', '\\&').replace('é', "\\'e"), panel4_results[k]) for k in scenarios.SCENARIOS]

np.random.seed(h.RNG_SEED)  # bootstrap CI
records = []
for group, cond, r in table_rows:
    lo, hi = flystat.resampling.bootstrap_confidence_intervals_from_data(np.array(r['r2']), use='median')
    records.append({'Group': group, 'Condition': cond, 'n': len(r['r2']),
                    'median': np.median(r['r2']), 'ci_lo': lo, 'ci_hi': hi})
table = pd.DataFrame(records)


def pooled(results):
    r2 = np.concatenate([r['r2'] for r in results])
    log10p = np.concatenate([r['log10p'] for r in results])
    return {'n': len(r2), 'r2_mean': r2.mean(), 'r2_sd': r2.std(ddof=1),
            'log10p_mean': log10p.mean(), 'log10p_sd': log10p.std(ddof=1)}

FICTIVE_ODOR_POOL = list(panel1_results.values()) + list(panel2_results.values())
CONTROL_POOL = [r for g in ['Var. wind sham (cm/s)', 'Sham', 'WT flash', 'WT sham'] for _, r, _ in panel3_results[g]]
fictive_odor_stats, control_stats = pooled(FICTIVE_ODOR_POOL), pooled(CONTROL_POOL)
for name, st in [('fictive odor', fictive_odor_stats), ('controls', control_stats)]:
    print('%-12s n = %5d  R2 = %.2f +/- %.2f  log10 p = %.1f +/- %.1f'
          % (name, st['n'], st['r2_mean'], st['r2_sd'], st['log10p_mean'], st['log10p_sd']))
table

fictive odor n =  3066  R2 = 0.86 +/- 0.18  log10 p = -25.3 +/- 8.8
controls     n =  2492  R2 = 0.63 +/- 0.30  log10 p = -18.9 +/- 11.4


,Group,Condition,n,median,ci_lo,ci_hi
0,Flash,Still air,200,0.911670,0.902236,0.920636
1,Flash,Unsteady,60,0.871550,0.828084,0.915730
2,Flash,Laminar (40 cm/s),145,0.872214,0.857006,0.889825
3,"Flash, variable wind",0 cm/s,700,0.947023,0.944022,0.949280
4,"Flash, variable wind",3 cm/s,491,0.926195,0.919852,0.933254
5,"Flash, variable wind",10 cm/s,257,0.903914,0.884395,0.914744
6,"Flash, variable wind",20 cm/s,700,0.904616,0.897947,0.912043
7,"Flash, variable wind",30 cm/s,272,0.879281,0.869807,0.892951
8,"Flash, variable wind",60 cm/s,241,0.895912,0.875756,0.909624
9,"Sham, variable wind",0 cm/s,388,0.782048,0.748056,0.806018


In [11]:
def stats_text(st):
    return (r'$R^2 = %.2f \pm %.2f$, $\log_{10} p = %.1f \pm %.1f$ ($n = %d$)'
            % (st['r2_mean'], st['r2_sd'], st['log10p_mean'], st['log10p_sd'], st['n']))


caption = (r'Median circular $R^2$ of each trajectory to its own best unifying-algorithm fit, with bootstrapped 95\% '
           r'confidence intervals of the median, scored over the region of interest of that fit. '
           r'Mean $\pm$ standard deviation over all trajectories: fictive odor (still air, unsteady, laminar and all '
           r'variable-wind speeds) ' + stats_text(fictive_odor_stats)
           + r'; controls (variable-wind sham, sham, WT flash and WT sham) ' + stats_text(control_stats)
           + r'. $p$ is from the Jammalamadaka--SenGupta circular correlation between each '
           r'trajectory and its own fit.')

lines = [r'\begin{table}[ht]', r'\centering', r'\caption{' + caption + '}', r'\label{tab:own_fit_r2}',
         r'\begin{tabular}{llrcc}', r'\hline',
         r'Group & Condition & $n$ & Median $R^2$ & 95\% CI \\', r'\hline']
prev_group = None
for _, row in table.iterrows():
    if prev_group is not None and row.Group != prev_group:
        lines.append(r'\hline')
    group = row.Group if row.Group != prev_group else ''
    lines.append(r'%s & %s & %d & %.2f & [%.2f, %.2f] \\' % (group, row.Condition, row.n, row['median'], row.ci_lo, row.ci_hi))
    prev_group = row.Group
lines += [r'\hline', r'\end{tabular}', r'\end{table}']

with open(TABLE_FILENAME, 'w') as f:
    f.write('\n'.join(lines) + '\n')
print('\n'.join(lines))

\begin{table}[ht]
\centering
\caption{Median circular $R^2$ of each trajectory to its own best unifying-algorithm fit, with bootstrapped 95\% confidence intervals of the median, scored over the region of interest of that fit. Mean $\pm$ standard deviation over all trajectories: fictive odor (still air, unsteady, laminar and all variable-wind speeds) $R^2 = 0.86 \pm 0.18$, $\log_{10} p = -25.3 \pm 8.8$ ($n = 3066$); controls (variable-wind sham, sham, WT flash and WT sham) $R^2 = 0.63 \pm 0.30$, $\log_{10} p = -18.9 \pm 11.4$ ($n = 2492$). $p$ is from the Jammalamadaka--SenGupta circular correlation between each trajectory and its own fit.}
\label{tab:own_fit_r2}
\begin{tabular}{llrcc}
\hline
Group & Condition & $n$ & Median $R^2$ & 95\% CI \\
\hline
Flash & Still air & 200 & 0.91 & [0.90, 0.92] \\
 & Unsteady & 60 & 0.87 & [0.83, 0.92] \\
 & Laminar (40 cm/s) & 145 & 0.87 & [0.86, 0.89] \\
\hline
Flash, variable wind & 0 cm/s & 700 & 0.95 & [0.94, 0.95] \\
 & 3 cm/s & 491 & 0.93 & [0.9